# 02 — Congestion Labelling and Time-Series Features

**Goal:** take the utilisation table from notebook 01 and produce the two things a
model needs:

1. **A label** — was each link Low / Medium / High / Critical at each moment?
2. **Features** — what the model is allowed to look at to predict that label.

The hard part is not the maths. It is that **the data has holes in it**, and the
obvious way to build lag features quietly gives wrong answers across those holes.
Most of this notebook is about doing that correctly.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path.cwd().parent
PROCESSED = PROJECT / "data" / "processed"

pd.set_option("display.width", 120)

# Utilisation table from notebook 01: one row per timestamp, one column per link.
util = pd.read_parquet(PROCESSED / "abilene_link_utilization_wide.parquet")
util.index = pd.DatetimeIndex(util.index)

print(f"{util.shape[0]:,} timesteps x {util.shape[1]} links")
print(f"{util.index.min():%Y-%m-%d} to {util.index.max():%Y-%m-%d}")
util.iloc[:3, :4].round(3)

48,096 timesteps x 30 links
2004-03-01 to 2004-09-10


link,"ATLA-M5,ATLAng","ATLAng,ATLA-M5","ATLAng,HSTNng","ATLAng,IPLSng"
2004-03-01 00:00:00,0.094,0.257,1.557,2.742
2004-03-01 00:05:00,0.095,0.366,1.769,2.685
2004-03-01 00:10:00,0.082,0.352,1.760,2.747


## Part 1 — Congestion labels

### Why not just use "% of capacity"?

The natural definition of congestion is something like *"over 80% of capacity"*.
That does not work here: Abilene was heavily over-provisioned, so the **average
link sits at 2.7%** utilisation. A fixed 80% threshold would put essentially every
observation in one class and leave the others empty — nothing to learn.

Instead we define congestion **relative to each link's own normal behaviour**:

| Class | Definition |
|---|---|
| Low | at or below that link's 50th percentile |
| Medium | between its 50th and 80th percentile |
| High | between its 80th and 95th percentile |
| Critical | above its 95th percentile |

So "Critical" means *"unusually busy **for this link**"*. A quiet link and a busy
link each get their own yardstick.

This also has a practical benefit: it never mentions capacity, so the same code
works on the GÉANT dataset, whose topology file has no capacities at all.

In [2]:
# .quantile() on the whole table gives per-column (per-link) percentiles in one go.
thresholds = util.quantile([0.50, 0.80, 0.95]).T
thresholds.columns = ["p50", "p80", "p95"]
thresholds.index.name = "link"

print("The busiest and quietest links have very different yardsticks:")
thresholds.sort_values("p95").iloc[[0, 1, -2, -1]].round(3)

The busiest and quietest links have very different yardsticks:


,p50,p80,p95
link,,,
"ATLA-M5,ATLAng",0.034,0.070,0.114
"ATLAng,ATLA-M5",0.061,0.118,0.202
"CHINng,IPLSng",4.937,6.474,8.763
"IPLSng,CHINng",5.224,6.977,9.081


In [3]:
CLASSES = ["Low", "Medium", "High", "Critical"]

# Count how many thresholds each value is above: 0 -> Low, 1 -> Medium,
# 2 -> High, 3 -> Critical. Comparing a DataFrame to a Series lines the Series
# up with the columns automatically, so this is one comparison per link.
codes = ((util > thresholds["p50"]).astype(int)
         + (util > thresholds["p80"]).astype(int)
         + (util > thresholds["p95"]).astype(int))

labels = pd.DataFrame(np.array(CLASSES)[codes.to_numpy()],
                      index=util.index, columns=util.columns)

print("labels for CHINng,IPLSng, first 5 timesteps:")
print(labels["CHINng,IPLSng"].head().to_string())

labels for CHINng,IPLSng, first 5 timesteps:
2004-03-01 00:00:00    Medium
2004-03-01 00:05:00       Low
2004-03-01 00:10:00       Low
2004-03-01 00:15:00    Medium
2004-03-01 00:20:00    Medium


In [4]:
distribution = (labels.stack().value_counts()
                .reindex(CLASSES)
                .to_frame("count"))
distribution["pct"] = (distribution["count"] / distribution["count"].sum() * 100).round(2)
distribution

,count,pct
Low,721440,50.0
Medium,432871,30.0
High,216419,15.0
Critical,72150,5.0


### Read this distribution carefully

It comes out at almost exactly 50 / 30 / 15 / 5 — but that is **not a discovery**.
We defined the classes by percentiles, so this split is guaranteed by construction.
If it came out any other way, it would mean a bug.

What is genuinely informative is *which* moments land in each class, and the
per-link thresholds above — the "Critical" line sits at 0.11% utilisation on the
quietest link and 9.08% on the busiest, a difference of about **80×**.

## Part 2 — Features, and the problem with gaps

### The problem

A lag feature means *"what was this link doing 5 minutes ago?"*. The obvious way
to build it is `util.shift(1)` — take the row above.

That is wrong here, because **the rows are not evenly spaced in time**. The 24
weekly files have four gaps in them, the largest 18 days. At the join, the row
above is not 5 minutes earlier — it is 18 days earlier.

### The fix

`.asfreq("5min")` rebuilds the table on a **complete, regular 5-minute grid** from
the first timestamp to the last. Timestamps we don't have data for still appear
as rows, filled with `NaN`.

Once the grid is regular, `.shift(1)` genuinely means "5 minutes ago", because
every row really is 5 minutes after the one above it.

### Why this is naturally gap-safe

This is the part worth being able to explain out loud:

> A lag that reaches back across a gap lands on a grid slot that has no data, so
> it picks up `NaN` and the feature is simply absent for that row. There is no
> gap-detection step, no list of gap boundaries, no special case — **the missing
> rows do the work**. Anything that tries to read across a hole finds the hole.

The same argument covers rolling windows. `.rolling(6)` requires 6 valid values by
default; if any of the 6 grid slots is empty, the result is `NaN`. So a rolling
average can never be quietly computed from six readings that are actually spread
across three weeks.

In [5]:
# The regular grid: every 5 minutes from the first timestamp to the last.
grid = util.asfreq("5min")

print(f"observations we actually have : {len(util):,}")
print(f"slots on the complete grid    : {len(grid):,}")
print(f"empty slots (the gaps)        : {grid.isna().any(axis=1).sum():,}")
print(f"\nthat is {grid.isna().any(axis=1).mean() * 100:.1f}% of the calendar span "
      "with no data")

observations we actually have : 48,096
slots on the complete grid    : 55,872
empty slots (the gaps)        : 7,776

that is 13.9% of the calendar span with no data


In [6]:
LAGS = [1, 2, 3, 6, 12, 288]        # 5m, 10m, 15m, 30m, 1h, 1 day
WINDOWS = [6, 12, 288]              # 30m, 1h, 1 day

# Build on the GRID (regular spacing), then keep only the rows we really observed.
lags = {n: grid.shift(n).loc[util.index] for n in LAGS}

# For rolling windows, shift by 1 FIRST so a row never includes its own value -
# that would be leakage: using the answer to predict the answer.
prior = grid.shift(1)

rolling = {}
for w in WINDOWS:
    rolling[("mean", w)] = prior.rolling(w).mean().loc[util.index]
    rolling[("std", w)] = prior.rolling(w).std().loc[util.index]

print(f"{len(lags)} lag features + {len(rolling)} rolling features")
print("\nmissing values per feature (caused by gaps and by the start of the series):")
for n in LAGS:
    print(f"  lag_{n:<5} {lags[n].isna().to_numpy().sum():>8,}")
for (stat, w) in rolling:
    print(f"  roll_{stat}_{w:<3} {rolling[(stat, w)].isna().to_numpy().sum():>8,}")

6 lag features + 6 rolling features

missing values per feature (caused by gaps and by the start of the series):
  lag_1          150
  lag_2          300
  lag_3          450
  lag_6          900
  lag_12       1,800
  lag_288     43,200
  roll_mean_6        900
  roll_std_6        900
  roll_mean_12     1,800
  roll_std_12     1,800
  roll_mean_288   43,200
  roll_std_288   43,200


### Cyclical time features

Network traffic follows the clock: quiet at 4am, busy in the afternoon, different
at weekends. So we give the model the hour of day and the day of week.

But we cannot hand it the raw number. If hour is `0…23`, then 23:00 and 00:00 look
23 apart when they are really 1 hour apart. The standard fix is to place the time
on a circle using sine and cosine, so midnight sits right next to 23:55.

These come straight from the timestamp, so gaps do not affect them — they are
always available.

In [7]:
hours = util.index.hour + util.index.minute / 60
days = util.index.dayofweek + hours / 24

cyclical = pd.DataFrame({
    "hour_sin": np.sin(2 * np.pi * hours / 24),
    "hour_cos": np.cos(2 * np.pi * hours / 24),
    "dow_sin": np.sin(2 * np.pi * days / 7),
    "dow_cos": np.cos(2 * np.pi * days / 7),
}, index=util.index)

# Check the wrap-around actually works: 23:55 and 00:00 should be close together.
a = cyclical.loc["2004-03-01 23:55", ["hour_sin", "hour_cos"]]
b = cyclical.loc["2004-03-02 00:00", ["hour_sin", "hour_cos"]]
print(f"distance on the circle between 23:55 and 00:00: {np.hypot(*(a - b)):.4f}")
print(f"(for comparison, 00:00 to 12:00 would be: 2.0)")
cyclical.head(3).round(4)

distance on the circle between 23:55 and 00:00: 0.0218
(for comparison, 00:00 to 12:00 would be: 2.0)


,hour_sin,hour_cos,dow_sin,dow_cos
2004-03-01 00:00:00,0.0000,1.0000,0.0000,1.0
2004-03-01 00:05:00,0.0218,0.9998,0.0031,1.0
2004-03-01 00:10:00,0.0436,0.9990,0.0062,1.0


## Part 3 — The gap example, side by side

This is the concrete demonstration that the grid approach works.

`X02` ends on **2004-03-14 23:55**. `X03` begins on **2004-04-02 00:00**. Between
them, 18 days are missing.

Look at the first row after that gap for the link `CHINng,IPLSng`, and compare
what the grid-based lag gives against what a naive `shift()` would have given.

In [8]:
LINK = "CHINng,IPLSng"
BEFORE_GAP = pd.Timestamp("2004-03-14 23:55")   # last row before the gap
AFTER_GAP = pd.Timestamp("2004-04-02 00:00")    # first row after the gap

print(f"link: {LINK}")
print(f"last reading before the gap : {BEFORE_GAP}  ->  {util.loc[BEFORE_GAP, LINK]:.4f}%")
print(f"first reading after the gap : {AFTER_GAP}  ->  {util.loc[AFTER_GAP, LINK]:.4f}%")
print(f"time actually elapsed       : {(AFTER_GAP - BEFORE_GAP).days} days")
print("\n...but a lag_1 feature is supposed to mean '5 minutes ago'.")

link: CHINng,IPLSng
last reading before the gap : 2004-03-14 23:55:00  ->  4.2517%
first reading after the gap : 2004-04-02 00:00:00  ->  6.0209%
time actually elapsed       : 18 days

...but a lag_1 feature is supposed to mean '5 minutes ago'.


In [9]:
# The naive version: shift by row position, ignoring the timestamps entirely.
naive_lags = {n: util.shift(n) for n in LAGS}

comparison = pd.DataFrame({
    "gap-aware (correct)": [lags[n].loc[AFTER_GAP, LINK] for n in LAGS],
    "naive shift (wrong)": [naive_lags[n].loc[AFTER_GAP, LINK] for n in LAGS],
}, index=[f"lag_{n}" for n in LAGS])

print(f"Features for {LINK} at {AFTER_GAP}:\n")
print(comparison.round(4).to_string())
print("\nThe naive column is not missing data - it is worse than missing. It is")
print(f"handing the model the reading from {BEFORE_GAP:%Y-%m-%d}, 18 days earlier,")
print("labelled as 'what happened 5 minutes ago'.")

Features for CHINng,IPLSng at 2004-04-02 00:00:00:

         gap-aware (correct)  naive shift (wrong)
lag_1                    NaN               4.2517
lag_2                    NaN               4.1757
lag_3                    NaN               3.9899
lag_6                    NaN               3.9150
lag_12                   NaN               4.0464
lag_288                  NaN               4.1368

The naive column is not missing data - it is worse than missing. It is
handing the model the reading from 2004-03-14, 18 days earlier,
labelled as 'what happened 5 minutes ago'.


In [10]:
# Same check for the rolling features.
roll_check = pd.DataFrame({
    "gap-aware (correct)": [rolling[(s, w)].loc[AFTER_GAP, LINK]
                            for (s, w) in rolling],
    "naive rolling (wrong)": [
        getattr(util.shift(1).rolling(w), s)().loc[AFTER_GAP, LINK]
        for (s, w) in rolling],
}, index=[f"roll_{s}_{w}" for (s, w) in rolling])

print(roll_check.round(4).to_string())

               gap-aware (correct)  naive rolling (wrong)
roll_mean_6                    NaN                 4.0498
roll_std_6                     NaN                 0.1414
roll_mean_12                   NaN                 3.9439
roll_std_12                    NaN                 0.1788
roll_mean_288                  NaN                 3.5407
roll_std_288                   NaN                 0.5813


In [11]:
# Pass/fail: every gap-aware feature at this row must be NaN, and the naive
# version must NOT be - otherwise the demonstration proves nothing.
gap_aware_all_nan = all(np.isnan(lags[n].loc[AFTER_GAP, LINK]) for n in LAGS)
naive_all_present = all(not np.isnan(naive_lags[n].loc[AFTER_GAP, LINK]) for n in LAGS)

print(f"[{'PASS' if gap_aware_all_nan else 'FAIL'}] every gap-aware lag is NaN at this row")
print(f"[{'PASS' if naive_all_present else 'FAIL'}] every naive lag has a (wrong) value")

# And one step later, inside the new block, the lag should work normally again.
one_step_later = AFTER_GAP + pd.Timedelta("5min")
recovered = lags[1].loc[one_step_later, LINK]
expected = util.loc[AFTER_GAP, LINK]
print(f"[{'PASS' if np.isclose(recovered, expected) else 'FAIL'}] "
      f"lag_1 recovers one step later ({recovered:.4f} == {expected:.4f})")

[PASS] every gap-aware lag is NaN at this row
[PASS] every naive lag has a (wrong) value
[PASS] lag_1 recovers one step later (6.0209 == 6.0209)


## Part 4 — How much does this actually matter?

Counting every (link, timestep, lag) value a naive implementation would have
filled in by reaching across a gap.

In [12]:
rows = []
for n in LAGS:
    correct_missing = int(lags[n].isna().to_numpy().sum())
    naive_missing = int(naive_lags[n].isna().to_numpy().sum())
    rows.append({
        "feature": f"lag_{n}",
        "gap-aware NaN": correct_missing,
        "naive NaN": naive_missing,
        "wrong values naive would give": correct_missing - naive_missing,
    })

impact = pd.DataFrame(rows).set_index("feature")
print(impact.to_string())
print(f"\nTotal values a naive approach would have got wrong: "
      f"{impact['wrong values naive would give'].sum():,}")

         gap-aware NaN  naive NaN  wrong values naive would give
feature                                                         
lag_1              150         30                            120
lag_2              300         60                            240
lag_3              450         90                            360
lag_6              900        180                            720
lag_12            1800        360                           1440
lag_288          43200       8640                          34560

Total values a naive approach would have got wrong: 37,440


## Part 5 — A note on scikit-learn, and on leakage

There is not much for scikit-learn to do yet — this notebook is data preparation,
and pandas covers it. Two honest notes:

- `sklearn.preprocessing.KBinsDiscretizer(strategy="quantile")` looks like it
  would do the labelling for us, but it splits into **equal-sized** bins. We need
  an uneven 50/30/15/5 split, so `.quantile()` is the right tool instead.
- `sklearn.model_selection.TimeSeriesSplit` **is** the right tool for splitting
  this data later, because it never puts future data in the training set.

That second point matters right now, for a reason that is easy to miss.

In [13]:
from sklearn.model_selection import TimeSeriesSplit

# TimeSeriesSplit always trains on earlier data and tests on later data.
splitter = TimeSeriesSplit(n_splits=3)

for i, (train_rows, test_rows) in enumerate(splitter.split(util), start=1):
    train_end = util.index[train_rows[-1]]
    test_start, test_end = util.index[test_rows[0]], util.index[test_rows[-1]]
    print(f"fold {i}: train up to {train_end:%Y-%m-%d}  |  "
          f"test {test_start:%Y-%m-%d} to {test_end:%Y-%m-%d}")

fold 1: train up to 2004-05-07  |  test 2004-05-07 to 2004-06-18
fold 2: train up to 2004-06-18  |  test 2004-06-18 to 2004-07-30
fold 3: train up to 2004-07-30  |  test 2004-07-30 to 2004-09-10


### The leakage we have *not* fixed yet

We computed the percentile thresholds using **the whole six months**, including
the part we will later use as a test set. That means the definition of "Critical"
already knows something about the future.

For labelling on its own that is fine — it is just how we define the target. But
before training a model, the thresholds must be recomputed **using training data
only**. The cell below shows the difference is real, not theoretical.

In [14]:
# Thresholds from the first fold's training window only...
train_rows, _ = next(iter(splitter.split(util)))
train_only = util.iloc[train_rows].quantile([0.50, 0.80, 0.95]).T
train_only.columns = ["p50", "p80", "p95"]

difference = (thresholds["p95"] - train_only["p95"]).abs().sort_values(ascending=False)

print("Largest differences in the 'Critical' threshold (full series vs train-only):")
print(pd.DataFrame({
    "full series": thresholds["p95"],
    "train only": train_only["p95"],
    "difference": thresholds["p95"] - train_only["p95"],
}).loc[difference.head(5).index].round(3).to_string())

print(f"\nmean absolute shift across all links: {difference.mean():.3f} "
      "percentage points")
print("\nSo yes - refit the thresholds on the training split before modelling.")

Largest differences in the 'Critical' threshold (full series vs train-only):
               full series  train only  difference
link                                              
IPLSng,KSCYng        7.457      33.305     -25.847
CHINng,IPLSng        8.763      34.045     -25.282
DNVRng,SNVAng        3.453      22.123     -18.670
KSCYng,DNVRng        6.220      24.130     -17.911
SNVAng,LOSAng        4.778      22.512     -17.734

mean absolute shift across all links: 3.901 percentage points

So yes - refit the thresholds on the training split before modelling.


### Why the difference is so large — worth understanding

The train-only "Critical" threshold is *four times higher* than the full-series
one on some links. That is not a rounding detail, so it is worth knowing what
causes it.

The average utilisation is fairly steady month to month. The **tail** is not.

In [15]:
monthly = pd.DataFrame({
    "mean": util.groupby(util.index.to_period("M")).mean().mean(axis=1),
    "p95 (CHINng,IPLSng)": util["CHINng,IPLSng"].groupby(
        util.index.to_period("M")).quantile(0.95),
    "max (CHINng,IPLSng)": util["CHINng,IPLSng"].groupby(
        util.index.to_period("M")).max(),
}).round(2)

print(monthly.to_string())

         mean  p95 (CHINng,IPLSng)  max (CHINng,IPLSng)
2004-03  2.74                 8.30                32.09
2004-04  3.24                52.83                59.37
2004-05  3.19                26.79                70.24
2004-06  2.49                 7.00                65.28
2004-07  2.08                 6.52                18.43
2004-08  2.30                 7.42                61.51
2004-09  3.54                 9.97                54.52


April and May contain sustained bursts of very high utilisation that the other
months simply do not have — the 95th percentile for `CHINng,IPLSng` is **52.8%**
in April but only **6.5%** in July, while the monthly *average* barely moves.

Two consequences to carry into modelling:

1. The first training fold is dominated by that unusual period, which is why its
   thresholds come out so high. Whichever window we fit thresholds on, we should
   say so explicitly rather than letting it happen by accident.
2. The busy months are not spread evenly through the six months, so a model
   trained on early data and tested on later data is being tested on a genuinely
   different traffic regime. That is realistic — real networks change — but it
   means we should not be surprised if accuracy on a time-ordered split is lower
   than on a shuffled one. The time-ordered number is the honest one.

## Part 6 — Assemble the feature table

One row per (timestamp, link), with the label and all 16 features.

`.to_numpy().ravel()` flattens a table row by row — all links for the first
timestamp, then all links for the second, and so on. Doing the same to every
feature table keeps every column lined up.

In [16]:
n_times, n_links = util.shape

features = pd.DataFrame({
    # each timestamp repeated once per link; link names tiled for each timestamp
    "timestamp": np.repeat(util.index, n_links),
    "link": np.tile(util.columns, n_times),
    "utilization_pct": util.to_numpy().ravel().astype("float32"),
    "congestion": pd.Categorical(labels.to_numpy().ravel(),
                                 categories=CLASSES, ordered=True),
})

for n in LAGS:
    features[f"lag_{n}"] = lags[n].to_numpy().ravel().astype("float32")

for (stat, w) in rolling:
    features[f"roll_{stat}_{w}"] = rolling[(stat, w)].to_numpy().ravel().astype("float32")

for column in cyclical.columns:
    features[column] = np.repeat(cyclical[column].to_numpy(), n_links).astype("float32")

print(f"{len(features):,} rows x {features.shape[1]} columns")
print(f"({n_times:,} timesteps x {n_links} links)")
features.head(3)

1,442,880 rows x 20 columns
(48,096 timesteps x 30 links)


,timestamp,link,utilization_pct,congestion,lag_1,lag_2,lag_3,lag_6,lag_12,lag_288,roll_mean_6,roll_std_6,roll_mean_12,roll_std_12,roll_mean_288,roll_std_288,hour_sin,hour_cos,dow_sin,dow_cos
0,2004-03-01,"ATLA-M5,ATLAng",0.093897,High,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,1.0
1,2004-03-01,"ATLAng,ATLA-M5",0.256962,Critical,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,1.0
2,2004-03-01,"ATLAng,HSTNng",1.557429,Low,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,1.0


In [17]:
feature_columns = [c for c in features.columns
                   if c not in ("timestamp", "link", "utilization_pct", "congestion")]

complete = features[feature_columns].notna().all(axis=1)
print(f"rows with every feature present : {complete.sum():,} ({complete.mean() * 100:.1f}%)")
print(f"rows with at least one missing  : {(~complete).sum():,} ({(~complete).mean() * 100:.1f}%)")
print("\nWe keep the rows with missing values rather than deleting them - XGBoost")
print("handles NaN natively, so the modelling step can decide what to do.")

rows with every feature present : 1,399,680 (97.0%)
rows with at least one missing  : 43,200 (3.0%)

We keep the rows with missing values rather than deleting them - XGBoost
handles NaN natively, so the modelling step can decide what to do.


## Part 7 — Sanity checks

In [18]:
checks = {
    "one row per (timestamp, link)":
        len(features) == n_times * n_links,
    "16 features built":
        len(feature_columns) == 16,
    "utilisation never missing":
        features["utilization_pct"].notna().all(),
    "every row has a label":
        features["congestion"].notna().all(),
    "labels are the four expected classes":
        list(features["congestion"].cat.categories) == CLASSES,
    "cyclical features never missing":
        features[["hour_sin", "hour_cos", "dow_sin", "dow_cos"]].notna().all().all(),
    "sin/cos stay on the unit circle":
        np.allclose(features["hour_sin"] ** 2 + features["hour_cos"] ** 2, 1.0, atol=1e-6),
    "no lag equals its own row (no leakage)":
        not (features["lag_1"] == features["utilization_pct"]).all(),
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

print(f"\n{sum(checks.values())}/{len(checks)} checks passed")

[PASS] one row per (timestamp, link)
[PASS] 16 features built
[PASS] utilisation never missing
[PASS] every row has a label
[PASS] labels are the four expected classes
[PASS] cyclical features never missing
[PASS] sin/cos stay on the unit circle
[PASS] no lag equals its own row (no leakage)

8/8 checks passed


## Part 8 — Verify against the previous pipeline

The old version of this project built these features with custom Python modules.
Before deleting those, check this notebook reproduces them **exactly**.

In [19]:
reference_path = PROCESSED / "abilene_features.parquet"

if reference_path.exists():
    reference = pd.read_parquet(reference_path)

    print(f"[{'PASS' if len(reference) == len(features) else 'FAIL'}] "
          f"row count      {len(features):,} vs {len(reference):,}")
    print(f"[{'PASS' if list(reference.columns) == list(features.columns) else 'FAIL'}] "
          f"column names   {features.shape[1]} columns, same order")
    print(f"[{'PASS' if reference['timestamp'].equals(features['timestamp']) else 'FAIL'}] "
          "timestamps     identical")
    print(f"[{'PASS' if (reference['link'] == features['link']).all() else 'FAIL'}] "
          "link column    identical")
    labels_match = (reference["congestion"].astype(str) ==
                    features["congestion"].astype(str)).all()
    print(f"[{'PASS' if labels_match else 'FAIL'}] "
          "congestion     every label identical")

    print("\nper-feature numeric comparison:")
    worst = 0.0
    for column in ["utilization_pct"] + feature_columns:
        old = reference[column].to_numpy()
        new = features[column].to_numpy()
        same_nan = np.array_equal(np.isnan(old), np.isnan(new))
        same_val = np.allclose(old, new, rtol=1e-6, atol=1e-6, equal_nan=True)
        gap = np.nanmax(np.abs(old - new)) if same_nan else float("nan")
        worst = max(worst, 0.0 if np.isnan(gap) else gap)
        print(f"  [{'PASS' if same_nan and same_val else 'FAIL'}] {column:<16}"
              f"NaN pattern {'same' if same_nan else 'DIFFERENT'}, "
              f"max diff {gap:.3e}")

    print(f"\nlargest difference across every feature and row: {worst:.3e}")
else:
    print("no reference file found - skipping comparison")

[PASS] row count      1,442,880 vs 1,442,880
[PASS] column names   20 columns, same order
[PASS] timestamps     identical
[PASS] link column    identical


[PASS] congestion     every label identical

per-feature numeric comparison:
  [PASS] utilization_pct NaN pattern same, max diff 0.000e+00
  [PASS] lag_1           NaN pattern same, max diff 0.000e+00
  [PASS] lag_2           NaN pattern same, max diff 0.000e+00
  [PASS] lag_3           NaN pattern same, max diff 0.000e+00


  [PASS] lag_6           NaN pattern same, max diff 0.000e+00
  [PASS] lag_12          NaN pattern same, max diff 0.000e+00
  [PASS] lag_288         NaN pattern same, max diff 0.000e+00
  [PASS] roll_mean_6     NaN pattern same, max diff 0.000e+00


  [PASS] roll_std_6      NaN pattern same, max diff 0.000e+00
  [PASS] roll_mean_12    NaN pattern same, max diff 0.000e+00
  [PASS] roll_std_12     NaN pattern same, max diff 0.000e+00
  [PASS] roll_mean_288   NaN pattern same, max diff 0.000e+00
  [PASS] roll_std_288    NaN pattern same, max diff 0.000e+00


  [PASS] hour_sin        NaN pattern same, max diff 0.000e+00
  [PASS] hour_cos        NaN pattern same, max diff 0.000e+00
  [PASS] dow_sin         NaN pattern same, max diff 0.000e+00
  [PASS] dow_cos         NaN pattern same, max diff 0.000e+00

largest difference across every feature and row: 0.000e+00


In [20]:
# Same for the thresholds.
reference_thresholds_path = PROCESSED / "abilene_congestion_thresholds.csv"

if reference_thresholds_path.exists():
    reference_thresholds = pd.read_csv(reference_thresholds_path, index_col="link")
    aligned = reference_thresholds.loc[thresholds.index]

    same = np.allclose(aligned.to_numpy(), thresholds.to_numpy(),
                       rtol=1e-12, atol=1e-12)
    print(f"[{'PASS' if same else 'FAIL'}] thresholds match for all "
          f"{len(thresholds)} links")
    print(f"largest difference: "
          f"{np.abs(aligned.to_numpy() - thresholds.to_numpy()).max():.3e}")

    print("\nsample (first 3 links):")
    display(pd.concat({"notebook": thresholds.head(3).round(4),
                       "old pipeline": aligned.head(3).round(4)}, axis=1))
else:
    print("no reference thresholds found - skipping comparison")

[PASS] thresholds match for all 30 links
largest difference: 8.882e-16

sample (first 3 links):


notebook                 old pipeline                
                    p50     p80     p95          p50     p80     p95
link                                                                
ATLA-M5,ATLAng   0.0339  0.0704  0.1138       0.0339  0.0704  0.1138
ATLAng,ATLA-M5   0.0609  0.1184  0.2018       0.0609  0.1184  0.2018
ATLAng,HSTNng    1.6366  2.2754  4.9461       1.6366  2.2754  4.9461

## Part 9 — Save

In [21]:
features.to_parquet(PROCESSED / "abilene_features.parquet", index=False)
thresholds.to_csv(PROCESSED / "abilene_congestion_thresholds.csv")

print(f"saved {len(features):,} rows x {features.shape[1]} columns")
print(f"saved thresholds for {len(thresholds)} links")
print("\nNext step: modelling - not started yet.")

saved 1,442,880 rows x 20 columns
saved thresholds for 30 links

Next step: modelling - not started yet.
